# Using Oracle Graph on 26ai with Jupyter Notebooks

### Prerequisite
1. Have the BANK_ACCOUNTS.csv and BANK_TRANSFERS.csv files in shared/datasets/bank_graph/data/


## 1. Install necessary packages

Install python-oracledb and prettytable packages

In [ ]:
## Optional
## install python-oracledb and prettytable packages in case they do not exist in the env
## use the python -m pip install approach into the current Jupyter kernel

import sys

!{sys.executable} -m pip install oracledb prettytable --upgrade

## 2. Connect to database using oracledb

The following paragraphs allow you to establish a connection to the database. Replace the username and password variables as needed.

In [ ]:
import oracledb
from prettytable import PrettyTable
import csv
import getpass
from pathlib import Path

data_candidates = [
    base / 'shared' / 'datasets' / 'bank_graph' / 'data'
    for base in [Path.cwd(), *Path.cwd().parents]
]
DATA_DIR = next((path for path in data_candidates if (path / 'BANK_ACCOUNTS.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not find shared/datasets/bank_graph/data. Run Jupyter from inside the repository.')

user = 'GRAPHUSER'
# pw = input('Enter Password for ' + user + ' user: ')
pw = getpass.getpass('Enter Password for ' + user + ' user: ')

### Connect to database

The following paragraph uses the user and pw variables from the previous paragraph to log into the database. Replace the dsn value with the connection string for your pluggable database. It should look something like the sample shown below.

In [ ]:
# Update this DSN with the connection string for your Oracle AI Database
oracledb.init_oracle_client()

connection = oracledb.connect(
    user=user,
    password=pw,
    dsn="""(DESCRIPTION=(CONNECT_TIMEOUT=5)(TRANSPORT_CONNECT_TIMEOUT=3)
            (RETRY_COUNT=3)(ADDRESS_LIST=(LOAD_BALANCE=on)(ADDRESS=(PROTOCOL=TCP)(HOST=<IP_Address>)
            (PORT=1521)))(CONNECT_DATA=(SERVICE_NAME=<PDB_Service_Name>)))
        """)

## 3. Create functions to execute database operations

The following paragraphs create functions to execute sql statements and commit them to the database, and execute sql queries and return them as a result set with the header values. 

In [ ]:
# Function to execute a sql statement (create, insert, update, drop)  
def execute_statement(statement):
    with connection.cursor() as cursor:
        cursor.execute(statement)

In [ ]:
# Function to execute a sql queries with the header and return the result set
def execute_query(query_string):
    with connection.cursor() as cursor:
        cursor.execute(query_string)
        columns = [col[0] for col in cursor.description]
        return [columns, cursor.fetchall()]

In [ ]:
# Function to verify if a table exists for the current user
def table_exists(tablename):
    with connection.cursor() as cursor:
        cursor.execute("""SELECT TABLE_NAME FROM USER_TABLES WHERE TABLE_NAME = :x1""", [tablename])
        return cursor.fetchone() is not None
        

## 4. Load dataset

Load the bank graph dataset using the oracledb library

In [ ]:
# Create BANK_ACCOUNTS table if it does not already exist
if not table_exists('BANK_ACCOUNTS'):
    s = '''CREATE TABLE BANK_ACCOUNTS (
                ID              NUMBER,
                NAME            VARCHAR(400),
                BALANCE         NUMBER(20,2)
            )'''
    execute_statement(s)
    
    # Load BANK_ACCOUNTS data from csv
    with open(DATA_DIR / 'BANK_ACCOUNTS.csv', newline='') as csvfile:
        csvreader = csv.reader(csvfile)
        bank_acct_data = [row for row in csvreader]
        with connection.cursor() as cursor:
            cursor.executemany('''INSERT INTO BANK_ACCOUNTS VALUES(:x1, :x2, :x3)''', bank_acct_data[1:])

In [ ]:
# Create BANK_TRANSFERS table if it does not already exist
if not table_exists('BANK_TRANSFERS'):
    s = '''CREATE TABLE BANK_TRANSFERS (
                TXN_ID          NUMBER,
                SRC_ACCT_ID     NUMBER,
                DST_ACCT_ID     NUMBER,
                DESCRIPTION     VARCHAR(400),
                AMOUNT          NUMBER
            )'''
    execute_statement(s)
    
    # Load BANK_TRANSFERS data from csv
    with open(DATA_DIR / 'BANK_TRANSFERS.csv', newline='') as csvfile:
        csvreader = csv.reader(csvfile)
        bank_transfer_data = [row for row in csvreader]
        with connection.cursor() as cursor:
            cursor.executemany('''INSERT INTO BANK_TRANSFERS VALUES(:x1, :x2, :x3, :x4, :x5)''', bank_transfer_data[1:])

In [ ]:
# Add constraints
constraints = ['ALTER TABLE BANK_ACCOUNTS ADD PRIMARY KEY (ID)', 'ALTER TABLE BANK_TRANSFERS ADD PRIMARY KEY (TXN_ID)', 'ALTER TABLE BANK_TRANSFERS MODIFY SRC_ACCT_ID REFERENCES BANK_ACCOUNTS (ID)', 'ALTER TABLE BANK_TRANSFERS MODIFY DST_ACCT_ID REFERENCES BANK_ACCOUNTS (ID)']

for constraint in constraints: 
    execute_statement(constraint)
    
    

## 5. Create property graph

Create the property graph by using the execute_statement function.

In [ ]:
# Create a property graph view on bank_accounts and bank_transfers
statement = """CREATE PROPERTY GRAPH bank_graph 
    VERTEX TABLES (
        BANK_ACCOUNTS
        KEY (ID)
        PROPERTIES (ID, Name, Balance) 
    )
    EDGE TABLES (
        BANK_TRANSFERS 
        KEY (TXN_ID) 
        SOURCE KEY (src_acct_id) REFERENCES BANK_ACCOUNTS(ID)
        DESTINATION KEY (dst_acct_id) REFERENCES BANK_ACCOUNTS(ID)
        PROPERTIES (src_acct_id, dst_acct_id, amount)
    ) """
execute_statement(statement)

## 6. Run SQL queries to gather information

In [ ]:
# This query shows the graphs available for the current user
query = """SELECT * FROM USER_PROPERTY_GRAPHS"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# This query shows the DDL for BANK_GRAPH
query = """SELECT DBMS_METADATA.GET_DDL('PROPERTY_GRAPH', 'BANK_GRAPH') FROM DUAL"""
rs = execute_query(query)
for obj in rs[1]:
    print(str(obj[0]))

In [ ]:
# This query shows the elements for BANK_GRAPH
query = """SELECT * FROM USER_PG_ELEMENTS WHERE GRAPH_NAME='BANK_GRAPH'"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# This query shows the labels and properties available in BANK_GRAPH
query = """SELECT GRAPH_NAME, LABEL_NAME, PROPERTY_NAME, DATA_LENGTH, DATA_CHAR_LENGTH, PROPERTY_ORDER
            FROM USER_PG_LABEL_PROPERTIES WHERE GRAPH_NAME='BANK_GRAPH'"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

## 7. Use SQL/PGQ queries to query BANK_GRAPH
**SQL/PGQ is a property graph query extension included in the SQL:2023 standard.**
It defines the CREATE PROPERTY GRAPH DDL, the GRAPH_TABLE() operator to query a graph, and the MATCH clause to specify the pattern to look for in the graph

In [ ]:
# Find the top 10 accounts by incoming transfers 
query = """SELECT ACCT_ID, COUNT(1) AS NUM_TRANSFERS
    FROM GRAPH_TABLE ( BANK_GRAPH
    MATCH (SRC) - [IS BANK_TRANSFERS] -> (DST)
    COLUMNS ( DST.ID AS ACCT_ID )
    ) GROUP BY ACCT_ID ORDER BY NUM_TRANSFERS DESC FETCH FIRST 10 ROWS ONLY"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# Find the top 10 accounts in the middle of a 2-hop chain of transfers
query = """SELECT ACCT_ID, COUNT(1) AS NUM_IN_MIDDLE
    FROM GRAPH_TABLE ( BANK_GRAPH
    MATCH (SRC) - [IS BANK_TRANSFERS] -> (VIA) - [IS BANK_TRANSFERS] -> (DST)
    COLUMNS ( VIA.ID AS ACCT_ID )
    ) GROUP BY ACCT_ID ORDER BY NUM_IN_MIDDLE DESC FETCH FIRST 10 ROWS ONLY"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# List accounts that received a transfer from account 387 in 1, 2, or 3 hops
query = """SELECT ACCOUNT_ID1, ACCOUNT_ID2 FROM GRAPH_TABLE(BANK_GRAPH
    MATCH (V1)-[IS BANK_TRANSFERS]->{1,3}(V2)
    WHERE V1.ID = 387
    COLUMNS (V1.ID AS ACCOUNT_ID1, V2.ID AS ACCOUNT_ID2))"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# Check if there are any 3-hop (triangles) transfers that start and end at the same account
query = """SELECT ACCT_ID, COUNT(1) AS NUM_TRIANGLES
    FROM GRAPH_TABLE (BANK_GRAPH
    MATCH (SRC) - []->{3} (SRC)
    COLUMNS (SRC.ID AS ACCT_ID)
    ) GROUP BY ACCT_ID ORDER BY NUM_TRIANGLES DESC"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# Check if there are any 4-hop transfers that start and end at the same account
query = """SELECT ACCT_ID, COUNT(1) AS NUM_4HOP_CHAINS
    FROM GRAPH_TABLE (BANK_GRAPH
    MATCH (SRC) - []->{4} (SRC)
    COLUMNS (SRC.ID AS ACCT_ID)
    ) GROUP BY ACCT_ID ORDER BY NUM_4HOP_CHAINS DESC"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# Check if there are any 5-hop transfers that start and end at the same account
query = """SELECT ACCT_ID, COUNT(1) AS NUM_5HOP_CHAINS
FROM GRAPH_TABLE (BANK_GRAPH
MATCH (SRC) - []->{5} (SRC)
COLUMNS (SRC.ID AS ACCT_ID)
) GROUP BY ACCT_ID ORDER BY NUM_5HOP_CHAINS DESC"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# List some (any 10) accounts which had a 3 to 5 hop circular payment chain 
query = """SELECT DISTINCT(ACCOUNT_ID)
FROM GRAPH_TABLE(BANK_GRAPH
   MATCH (V1)-[IS BANK_TRANSFERS]->{3,5}(V1)
    COLUMNS (V1.ID AS ACCOUNT_ID)
) FETCH FIRST 10 ROWS ONLY"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# Query accounts by number of 3 to 5 hops cycles in descending order. Show top 10. 
query = """SELECT DISTINCT(ACCOUNT_ID), COUNT(1) AS NUM_CYCLES FROM GRAPH_TABLE(BANK_GRAPH
    MATCH (V1)-[IS BANK_TRANSFERS]->{3, 5}(V1)
    COLUMNS (V1.ID AS ACCOUNT_ID) )
    GROUP BY ACCOUNT_ID ORDER BY NUM_CYCLES DESC FETCH FIRST 10 ROWS ONLY"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

In [ ]:
# List all accounts in a 5 hop (or 6 hop) circular chain 
query = """SELECT START_ACCT, FIRST_ACCT, SECOND_ACCT, THIRD_ACCT, FOURTH_ACCT, FIFTH_ACCT, START_ACCT AS END_IN_ACCT
            FROM GRAPH_TABLE(BANK_GRAPH
            MATCH (SRC) -[IS BANK_TRANSFERS]->(F) -[IS BANK_TRANSFERS]-> (S)
            -[IS BANK_TRANSFERS]-> (T) -[IS BANK_TRANSFERS]-> (FO)
            -[IS BANK_TRANSFERS]-> (FI) -[IS BANK_TRANSFERS]->(SRC)
        WHERE SRC.ID=934
        COLUMNS (
        SRC.ID AS START_ACCT,
        F.ID AS FIRST_ACCT,
        S.ID AS SECOND_ACCT,
        T.ID AS THIRD_ACCT,
        FO.ID AS FOURTH_ACCT,
        FI.ID AS FIFTH_ACCT)
        )"""
rs = execute_query(query)
table = PrettyTable()
table.field_names = rs[0]

for row in rs[1]:
    table.add_row(row)
    
print(table)

## 8. Commit and close connection 

In [ ]:
## Optional: commit if you want data to persist
connection.commit()

In [ ]:
connection.close()